In [1]:
from pathlib import Path

import torch

from rtnls_inference import (
    HeatmapRegressionEnsemble,
    SegmentationEnsemble,
)

## Segmentation of preprocessed images

Here we segment images preprocessed using 0_preprocess.ipynb


In [2]:
ds_path = Path("../samples/fundus")

# input folders. these are the folders where we stored the preprocessed images
rgb_path = ds_path / "rgb"
ce_path = ds_path / "ce"

# these are the output folders for:
av_path = ds_path / "av"  # artery-vein segmentations
vessels_path = ds_path / "vessels"  # vessels segmentation
discs_path = ds_path / "discs"  # optic disc segmentations
overlays_path = ds_path / "overlays"  # optional overlay visualizations

device = torch.device("cuda:0")  # device to use for inference

In [3]:
rgb_paths = sorted(list(rgb_path.glob("*.png")))
ce_paths = sorted(list(ce_path.glob("*.png")))
paired_paths = list(zip(rgb_paths, ce_paths))

In [4]:
paired_paths[0]  # important to make sure that the paths are paired correctly

(PosixPath('../samples/fundus/rgb/CHASEDB1_08L.png'),
 PosixPath('../samples/fundus/ce/CHASEDB1_08L.png'))

### Artery-vein segmentation


In [5]:
av_ensemble = SegmentationEnsemble.from_release("av_july24.pt").to(device)
av_ensemble.predict_preprocessed(paired_paths, dest_path=av_path, num_workers=2)

FundusTestTransform initialized with contrast_enhance: False


  0%|          | 0/1 [00:00<?, ?it/s]/home/jose/.pyenv/versions/3.12.11/envs/rtnls/lib/python3.12/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/home/jose/.pyenv/versions/3.12.11/envs/rtnls/lib/python3.12/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a diff

## Vessel segmentation (optional)

In [6]:
vessels_ensemble = SegmentationEnsemble.from_release("vessels_july24.pt").to(device)
vessels_ensemble.predict_preprocessed(paired_paths, dest_path=vessels_path, num_workers=2)

FundusTestTransform initialized with contrast_enhance: False


100%|██████████| 1/1 [00:07<00:00,  7.44s/it]


### Disc segmentation


In [7]:
disc_ensemble = SegmentationEnsemble.from_release("disc_july24.pt").to(device)
disc_ensemble.predict_preprocessed(paired_paths, dest_path=discs_path, num_workers=2)

FundusTestTransform initialized with contrast_enhance: False


100%|██████████| 1/1 [00:02<00:00,  2.93s/it]


### Fovea detection


In [8]:
fovea_ensemble = HeatmapRegressionEnsemble.from_release("fovea_july24.pt").to(device)
# note: this model does not use contrast enhanced images
df = fovea_ensemble.predict_preprocessed(paired_paths, num_workers=2)
df.columns = ["mean_x", "mean_y"]
df.to_csv(ds_path / "fovea.csv")

FundusTestTransform initialized with contrast_enhance: False


  0%|          | 0/1 [00:00<?, ?it/s]/home/jose/.pyenv/versions/3.12.11/envs/rtnls/lib/python3.12/site-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
100%|██████████| 1/1 [00:02<00:00,  2.68s/it]


In [9]:
df

,mean_x,mean_y
CHASEDB1_08L,953.400024,553.799988
CHASEDB1_12R,101.800003,634.200012
DRIVE_22,486.600006,522.599976
DRIVE_40,503.399994,521.000000
HRF_04_g,551.000000,535.400024
HRF_07_dr,545.000000,534.599976


### Plotting the retinas (optional)

This will only work if you ran all the models and stored the outputs using the same folder/file names as above


In [ ]:
from vascx.fundus import RetinaLoader

from rtnls_enface.utils.plotting import plot_gridfns

loader = RetinaLoader.from_folder(ds_path)

In [ ]:
plot_gridfns([ret.plot for ret in loader[:6]])

### Storing visualizations (optional)


In [ ]:
if not overlays_path.exists():
    overlays_path.mkdir()
for ret in loader:
    fig, _ = ret.plot()
    fig.savefig(overlays_path / f"{ret.id}.png", bbox_inches="tight", pad_inches=0)